# Ejercicio 03 · Corpus público y clasificador de texto reproducible

**Autora:** Kimberly Pereyra · INF-8239 Ciencia de Datos II · Unidad 02

- **Pregunta:** ¿Puede un clasificador basado en TF-IDF distinguir oraciones positivas y negativas de reseñas de productos, películas y restaurantes, y qué tipo de oraciones lo confunden?
- **Corpus:** *Sentiment Labelled Sentences* — UCI ML Repository, id 331 (Kotzias, 2015). DOI 10.24432/C57604. Licencia CC BY 4.0.
- **Unidad de análisis:** una oración de una reseña. **Target:** `label` (negative / positive).
- **Métrica principal:** F1 macro (ambas clases pesan igual).

Este notebook ejecuta los mismos scripts del proyecto (`scripts/`) y muestra sus resultados. Antes de ejecutarlo: `uv run python scripts/download_data.py`.

In [ ]:
import json
import runpy
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from inf8239_u02.config import ROOT, settings
from inf8239_u02.evaluation import merge_error_review

REPORTS = ROOT / "reports"
pd.set_option("display.max_colwidth", 120)

def run_script(name):
    runpy.run_path(str(ROOT / "scripts" / name), run_name="__main__")

print("Dataset:", settings.dataset_path.relative_to(ROOT), "| existe:", settings.dataset_path.exists())

## 1. Auditoría del corpus

In [ ]:
run_script("audit_data.py")
display(Image(REPORTS / "audit_corpus.png"))

**Observación → evidencia → interpretación → decisión**

- **Observación:** 3,000 oraciones, 1,000 por fuente, con 500 positivas y 500 negativas en cada una. No hay nulos ni textos vacíos.
- **Evidencia:** `reports/audit_summary.json` registra 18 duplicados exactos y 21 al normalizar mayúsculas y espacios, todos con la misma etiqueta (0 conflictos). La mediana es de 10 palabras y el percentil 99 de 36.
- **Interpretación:** el corpus está balanceado, así que accuracy no es engañosa, pero se reporta F1 macro. Los duplicados, aunque pocos, podrían quedar en entrenamiento y en prueba e inflar el resultado. Las oraciones son cortas, lo que deja poco contexto.
- **Decisión:** eliminar los 21 duplicados **antes** de dividir (quedan 2,979) y usar una única partición estratificada por clase y fuente, con semilla 42.

## 2. Entrenamiento: baseline y dos pipelines comparables

In [ ]:
run_script("train_text.py")

In [ ]:
metrics = pd.read_csv(REPORTS / "text_metrics.csv")
cols = ["model", "f1_macro", "accuracy", "recall_negative", "recall_positive", "fit_seconds", "predict_ms", "vocabulary"]
display(metrics[cols].round(4))
by_source = pd.read_csv(REPORTS / "text_metrics_by_source.csv")
display(by_source.pivot(index="model", columns="source", values="f1_macro").round(3))
print(json.loads((REPORTS / "selected_model.json").read_text()))

- **Prevención de fuga:** los tres modelos usan el mismo `Pipeline(TfidfVectorizer → modelo)`, así que el vocabulario y los pesos IDF se aprenden solo con el 75 % de entrenamiento. El script verifica que no haya textos compartidos entre entrenamiento y prueba.
- **Baseline:** `DummyClassifier` (siempre la clase más frecuente) obtiene F1 macro ≈ 0.33 y recall 0 en una de las clases.
- **Comparación:** Complement Naive Bayes y Regresión logística quedan muy cerca (≈ 0.83). La diferencia es menor a un punto porcentual, por lo que no es concluyente con 745 oraciones de prueba.
- **Decisión:** se conserva la regresión logística por tener el F1 más alto y porque entrega probabilidades, útiles para revisar la confianza de los errores. Ambas superan ampliamente el baseline.
- **Por fuente:** IMDb obtiene el mejor F1 y Yelp el más bajo; el desempeño cambia según el dominio.

## 3. Matriz de confusión

In [ ]:
display(Image(REPORTS / "confusion_text.png"))

- La diagonal concentra los aciertos. Las celdas fuera de la diagonal son parecidas en ambas direcciones: el modelo no favorece una clase.
- **Costo de la confusión:** en un uso real (por ejemplo, detectar clientes insatisfechos), una oración negativa clasificada como positiva es el error más costoso, porque esa queja no se atendería.

## 4. Análisis de 20 errores

In [ ]:
sample = pd.read_csv(REPORTS / "error_analysis_20.csv")
review = pd.read_csv(ROOT / "docs" / "error_review.csv")
reviewed = merge_error_review(sample, review)
assert (reviewed["category"] != "SIN REVISAR").all(), "Hay errores sin categoría: revise docs/error_review.csv"
reviewed.to_csv(REPORTS / "error_analysis_20_reviewed.csv", index=False)
display(reviewed[["text", "source", "real", "predicted", "confidence", "category"]])

counts = reviewed["category"].value_counts()
ax = counts.sort_values().plot.barh(figsize=(7, 3.5), title="Categorías de los 20 errores analizados")
ax.set_xlabel("Errores")
plt.tight_layout(); plt.savefig(REPORTS / "error_categories.png", dpi=160); plt.show()
counts

- **Observación:** de 125 errores en prueba, se analizaron 20 elegidos al azar (semilla 42). La mayoría tiene una confianza cercana a 0.5, es decir, el modelo duda en esos casos.
- **Evidencia:** las categorías más frecuentes son *sentimiento mixto* (contrastes con *but* o *even if*), *negación* (*won't be revisiting*, *couldn't have been simpler*), *texto corto o implícito* y *expresiones idiomáticas o jerga* (*killer*, *melt in your mouth*). También aparece un caso de ironía.
- **Interpretación:** TF-IDF trata cada palabra o par de palabras por separado, así que no entiende negaciones, contrastes ni sentido figurado.
- **Hipótesis para el siguiente experimento:** agregar n-gramas que capturen la negación o usar embeddings o modelos contextuales, y comparar en los mismos 20 casos.

## 5. Verificaciones mínimas

In [ ]:
import joblib
model = joblib.load(ROOT / "models" / "text_model.joblib")
examples = ["The food was great and the staff very friendly.", "The battery died after two days, terrible."]
pred = model.predict(examples)
print(dict(zip(examples, pred)))
assert set(pred) <= {"negative", "positive"}
assert "tfidf" in model.named_steps and "model" in model.named_steps
print("Verificaciones superadas")

## Cierre interpretativo

- **Resultado principal:** la regresión logística con TF-IDF alcanza un F1 macro ≈ 0.83 frente a ≈ 0.33 del baseline; Naive Bayes logra un resultado casi igual.
- **Modelo seleccionado y evidencia:** regresión logística (mejor F1 por un margen pequeño y probabilidades disponibles); ver `reports/text_metrics.csv`.
- **Clase con mayor dificultad:** ninguna clase destaca; los recall de ambas clases son parecidos. La dificultad está en el **dominio** (Yelp) y en el **tipo de oración**.
- **Tipo de error más frecuente:** sentimiento mixto y negación.
- **Impacto en el contexto:** una queja con negación o ironía podría pasar como positiva y quedar sin atención.
- **Limitación del dataset:** inglés, tres sitios web y oraciones seleccionadas por tener un sentimiento claro (sin neutrales). No se puede aplicar a mensajes en español dominicano.
- **Decisión antes del despliegue:** usarlo solo como apoyo con revisión humana, mostrar la confianza y evaluar primero con datos del dominio real.